In [5]:
from pathlib import Path
import wave

import pandas as pd
from datasets import Audio, Dataset

root = Path("data") if Path("data/karakalpak-audio-dataset-hf/data.csv").exists() else Path("../data")
hf_dir = root / "karakalpak-audio-dataset-hf"
frame = pd.read_csv(hf_dir / "data.csv")
if frame["audio_index"].tolist() != list(range(len(frame))):
    raise ValueError("data.csv audio_index must be sequential")
 
def sample_rate(path):
    with wave.open(str(path), "rb") as audio:
        return audio.getframerate()

data_files = sorted((hf_dir / "data").glob("*.wav"), key=lambda path: (sample_rate(path), path.name))
audio_16k_by_name = {path.name: path for path in (hf_dir / "data_16k").glob("*.wav")}
if frame["filename"].tolist() != [path.name for path in data_files] or set(frame["filename"]) != set(audio_16k_by_name):
    raise ValueError("data.csv rows must match both audio folders")
frame["audio_original"] = [{"path": str(path.resolve()), "bytes": None} for path in data_files]
frame["audio"] = [{"path": str(audio_16k_by_name[name].resolve()), "bytes": None} for name in frame["filename"]]
dataset = Dataset.from_pandas(frame, preserve_index=False).cast_column("audio_original", Audio(decode=False)).cast_column("audio", Audio(decode=False))
dataset

Dataset({
    features: ['audio_index', 'filename', 'sentence_latin', 'sentence_cyrillic', 'duration', 'audio_original', 'audio'],
    num_rows: 3032
})

In [6]:
print(dataset)

Dataset({
    features: ['audio_index', 'filename', 'sentence_latin', 'sentence_cyrillic', 'duration', 'audio_original', 'audio'],
    num_rows: 3032
})


In [7]:
import pandas as pd 
import numpy as np

In [8]:
data = pd.read_csv(hf_dir / "data.csv")
data.head()

,audio_index,filename,sentence_latin,sentence_cyrillic,duration
0,0,kaa_audio_1000001.wav,Internatta,Интернатта,1.582625
1,1,kaa_audio_1000002.wav,Oktyabr ayınıń sońǵı kúnleri.,Октябрь айының соңғы күнлери.,2.967375
2,2,kaa_audio_1000003.wav,"Kún uyasına sıymay dolanıp, qıp-qızıl shoqtay ...","Күн уясына сыймай доланып, қып-қызыл шоқтай жа...",6.952250
3,3,kaa_audio_1000004.wav,“Kese etek”ten Qońıratqa qaray sozılǵan úlken ...,“Кесе етек”тен Қоңыратқа қарай созылған үлкен ...,5.058687
4,4,kaa_audio_1000005.wav,Jeter jerde el joq.,Жетер жерде ел жоқ.,2.006562
